In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier

In [2]:
chemins = [
    Path("data/raw/student_mental_health.csv"),
    Path("../data/raw/student_mental_health.csv"),
    Path("student_mental_health.csv")
]

chemin_csv = next((p for p in chemins if p.exists()), None)
if chemin_csv is None:
    raise FileNotFoundError("Placez student_mental_health.csv dans data/raw/")

df = pd.read_csv(chemin_csv)
print("Dimensions :", df.shape)
display(df.head(3))

Dimensions : (2000, 17)


,Student_ID,Age,Gender,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Mental_Health_Risk
0,STU0001,22,Male,3,6.6,17.6,2.91,6.9,7.5,Moderate,8.5,4.0,78.0,No,Poor,No,High
1,STU0002,21,Male,4,5.6,22.4,2.92,7.6,11.5,Low,4.5,5.3,85.4,Yes,Average,No,Medium
2,STU0003,22,Male,5,7.1,28.1,2.68,6.5,8.9,Low,4.8,6.1,79.5,No,Average,No,Medium


In [3]:
cible = "Mental_Health_Risk"
df = df.dropna(subset=[cible]).copy()

X = df.drop(columns=["Student_ID", cible])
y = df[cible]

print("Nombre de variables X :", X.shape[1])
print("Classes :", sorted(y.unique()))
print("Nombre d'étudiants par classe :")
print(y.value_counts())

Nombre de variables X : 15
Classes : ['High', 'Low', 'Medium']
Nombre d'étudiants par classe :
Mental_Health_Risk
High      680
Medium    660
Low       660
Name: count, dtype: int64


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

dictionnaire_classes = {"Low": 0, "Medium": 1, "High": 2}
y_train = y_train.map(dictionnaire_classes)
y_test = y_test.map(dictionnaire_classes)

print("Train :", X_train.shape)
print("Test  :", X_test.shape)
print("Répartition du train :")
print(y_train.value_counts().sort_index())

Train : (1600, 15)
Test  : (400, 15)
Répartition du train :
Mental_Health_Risk
0    528
1    528
2    544
Name: count, dtype: int64


In [5]:
colonnes_num = X_train.select_dtypes(include="number").columns.tolist()
colonnes_cat = X_train.select_dtypes(exclude="number").columns.tolist()

print("Numériques :", colonnes_num)
print("Catégorielles :", colonnes_cat)

Numériques : ['Age', 'Year_of_Study', 'Sleep_Hours', 'Study_Hours_Per_Week', 'GPA', 'Social_Media_Hours', 'Screen_Time_Hours', 'Financial_Stress', 'Family_Support', 'Attendance_Rate']
Catégorielles : ['Gender', 'Physical_Activity', 'Part_Time_Job', 'Diet_Quality', 'Previous_Mental_Health_History']


In [6]:
class IQRClipper(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        q1 = X.quantile(0.25)
        q3 = X.quantile(0.75)
        iqr = q3 - q1
        self.basse_ = q1 - 1.5 * iqr
        self.haute_ = q3 + 1.5 * iqr
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        return X.clip(self.basse_, self.haute_, axis=1).to_numpy()

In [7]:
pipeline_num = Pipeline([
    ("imputation", SimpleImputer(strategy="median")),
    ("iqr", IQRClipper()),
    ("standardisation", StandardScaler())
])

pipeline_cat = Pipeline([
    ("imputation", SimpleImputer(strategy="most_frequent")),
    ("encodage", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", pipeline_num, colonnes_num),
    ("cat", pipeline_cat, colonnes_cat)
])

print("Prétraitement prêt : imputation + IQR + encodage + standardisation")

Prétraitement prêt : imputation + IQR + encodage + standardisation


In [8]:
modeles = {
    "Random Forest": RandomForestClassifier(
        n_estimators=100, random_state=42, n_jobs=2
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100, max_depth=3, learning_rate=0.1,
        objective="multi:softprob", eval_metric="mlogloss",
        random_state=42, n_jobs=2
    ),
    "SVM": SVC(kernel="rbf", C=1.0)
}

print("Modèles à comparer :", list(modeles))

Modèles à comparer : ['Random Forest', 'XGBoost', 'SVM']


In [9]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

metriques = {
    "accuracy": "accuracy",
    "precision": "precision_macro",
    "recall": "recall_macro",
    "f1": "f1_macro"
}

resultats = []
pipelines_entraines = {}

for nom, modele in modeles.items():
    pipeline = Pipeline([
        ("preparation", clone(preprocessor)),
        ("modele", clone(modele))
    ])

    scores = cross_validate(
        pipeline, X_train, y_train,
        cv=cv, scoring=metriques, n_jobs=1
    )

    pipeline.fit(X_train, y_train)
    pipelines_entraines[nom] = pipeline

    resultats.append({
        "Modèle": nom,
        "Accuracy": scores["test_accuracy"].mean(),
        "Precision_macro": scores["test_precision"].mean(),
        "Recall_macro": scores["test_recall"].mean(),
        "F1_macro": scores["test_f1"].mean()
    })

comparaison = pd.DataFrame(resultats).sort_values("F1_macro", ascending=False)
display(comparaison.round(3))

,Modèle,Accuracy,Precision_macro,Recall_macro,F1_macro
2,SVM,0.716,0.718,0.716,0.717
1,XGBoost,0.687,0.682,0.686,0.683
0,Random Forest,0.673,0.662,0.672,0.664
